# Part II · Lesson 4 — Look up an employee's leave balance

### Import the graph helpers

In [ ]:
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END

### Import the model and messages

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

### Connect to our existing Groq model

In [ ]:
# Keep GROQ_API_KEY in the local .env file.
load_dotenv()
model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
    reasoning_effort="low",
)

### Import the tool helpers

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition

### Keep the employee and conversation in state

In [ ]:
class LeaveState(TypedDict):
    employee_id: str
    request_type: str
    messages: Annotated[list, add_messages]
    status: str

### Expose the leave-record lookup

In [ ]:
@tool
def get_leave_balance(employee_id: str) -> dict:
    """Read the paid leave allowance, days used, and balance for an employee."""
    records = {
        "EMP101": {"allowance": 20, "used": 14, "remaining": 6},
        "EMP102": {"allowance": 20, "used": 8, "remaining": 12},
    }
    return records[employee_id]

### Make the tool available to the model

In [ ]:
leave_tools = [get_leave_balance]
tool_model = model.bind_tools(leave_tools)

### Ask the model for the required information

In [ ]:
def ask_model(state: LeaveState) -> dict:
    reply = tool_model.invoke([
        SystemMessage(
            f"Answer a {state['request_type']} request for {state['employee_id']}. "
            "Use get_leave_balance for the balance; do not guess it."
        ),
        *state["messages"],
    ])
    return {"messages": [reply], "status": "model replied"}

### Write the answer from the tool result

In [ ]:
def answer_from_tool(state: LeaveState) -> dict:
    reply = model.invoke([
        SystemMessage("Explain the employee's leave balance using the tool result."),
        *state["messages"],
    ])
    return {"messages": [reply], "status": "answered"}

### Register the model and execution nodes

In [ ]:
builder = StateGraph(LeaveState)
builder.add_node("ask_model", ask_model)
builder.add_node("tools", ToolNode(leave_tools))
builder.add_node("answer", answer_from_tool)

### Connect one controlled tool round

In [ ]:
builder.add_edge(START, "ask_model")
builder.add_conditional_edges(
    "ask_model", tools_condition, {"tools": "tools", END: END}
)
builder.add_edge("tools", "answer")
builder.add_edge("answer", END)
graph = builder.compile()

### Ask about the selected employee

In [ ]:
leave_input = {
    "employee_id": "EMP101", "request_type": "paid leave",
    "messages": [HumanMessage("How many paid leave days do I have left?")],
    "status": "new",
}
response = graph.invoke(leave_input)
print(response["messages"][-1].text)

### Inspect the request and result

In [ ]:
for message in response["messages"]:
    print(message.type, message.text)
    if getattr(message, "tool_calls", None):
        print("Requested calls:", message.tool_calls)